In [1]:
import csv
import os


def assign_grade(avg):
    if avg >= 85:
        return "A"
    elif avg >= 70:
        return "B"
    elif avg >= 55:
        return "C"
    elif avg >= 40:
        return "D"
    else:
        return "F"


def build_gradebook(
    students_path="students.csv",
    scores_path="scores.csv",
    report_path="report.csv"
):
    if not os.path.exists(students_path):
        raise FileNotFoundError(f"Students file nahi mili: {students_path}")
    if not os.path.exists(scores_path):
        raise FileNotFoundError(f"Scores file nahi mili: {scores_path}")

    students = {}

    # Step 1: students.csv read karein
    with open(students_path, "r", newline="", encoding="utf-8") as s_file:
        reader = csv.DictReader(s_file)
        if not reader.fieldnames or not {"student_id", "name"}.issubset(set(reader.fieldnames)):
            raise ValueError("students.csv mein student_id aur name columns hone chahiye")

        for row in reader:
            sid = (row.get("student_id") or "").strip()
            name = (row.get("name") or "").strip()

            if not sid or not name:
                raise ValueError("students.csv mein khali student_id ya name mila")
            if sid in students:
                raise ValueError(f"Duplicate student_id mila: {sid}")

            students[sid] = {
                "student_id": sid,
                "name": name,
                "marks": []
            }

    seen_pairs = set()

    # Step 2: scores.csv read aur validate karein
    with open(scores_path, "r", newline="", encoding="utf-8") as sc_file:
        reader = csv.DictReader(sc_file)
        if not reader.fieldnames or not {"student_id", "subject", "marks"}.issubset(set(reader.fieldnames)):
            raise ValueError("scores.csv mein student_id, subject, aur marks columns hone chahiye")

        for row in reader:
            sid = (row.get("student_id") or "").strip()
            subject = (row.get("subject") or "").strip()
            marks_raw = (row.get("marks") or "").strip()

            # Unknown student ID reject karein
            if sid not in students:
                raise ValueError(f"Unknown student_id mila scores.csv mein: '{sid}'")

            if not subject:
                raise ValueError(f"Student '{sid}' ke liye subject khali hai")

            # Duplicate (student_id, subject) reject karein (case-insensitive check)
            pair_key = (sid, subject.lower())
            if pair_key in seen_pairs:
                raise ValueError(f"Duplicate entry mili: student '{sid}', subject '{subject}'")
            seen_pairs.add(pair_key)

            # Marks 0 se 100 ke darmiyan valid number hone chahiye
            try:
                marks = float(marks_raw)
            except ValueError:
                raise ValueError(f"Invalid marks '{marks_raw}' student '{sid}' ke liye")

            if marks < 0 or marks > 100:
                raise ValueError(f"Marks 0-100 range se bahar hain ({marks}) student '{sid}' ke liye")

            students[sid]["marks"].append(marks)

    # Step 3: Har student ki average aur grade calculate karein
    records = []
    for sid, info in students.items():
        marks_list = info["marks"]
        if len(marks_list) == 0:
            avg = 0.0
        else:
            avg = round(sum(marks_list) / len(marks_list), 2)

        grade = assign_grade(avg)
        records.append({
            "student_id": sid,
            "name": info["name"],
            "average": avg,
            "grade": grade
        })

    # Step 4: Sort karein -> highest average pehle, tie hone par student_id ascending
    # Agar ID numeric ho (jaise "101", "102") to numeric order, warna string order
    def sort_key(item):
        sid = item["student_id"]
        id_key = (0, int(sid)) if sid.isdigit() else (1, sid)
        return (-item["average"], id_key)

    records.sort(key=sort_key)

    # Rank assign karein (1 se start)
    for i in range(len(records)):
        records[i]["rank"] = i + 1

    # Step 5: report.csv write karein
    with open(report_path, "w", newline="", encoding="utf-8") as rep_file:
        fieldnames = ["rank", "student_id", "name", "average", "grade"]
        writer = csv.DictWriter(rep_file, fieldnames=fieldnames)
        writer.writeheader()
        for rec in records:
            writer.writerow(rec)

    return records


# Sample data generation & testing
if __name__ == "__main__":
    with open("students.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["student_id", "name"])
        writer.writerow(["101", "Haider"])
        writer.writerow(["102", "Ayesha"])
        writer.writerow(["103", "Bilal"])

    with open("scores.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["student_id", "subject", "marks"])
        writer.writerow(["101", "Math", 90])
        writer.writerow(["101", "Physics", 80])
        writer.writerow(["101", "Chemistry", 85])
        writer.writerow(["102", "Math", 85])
        writer.writerow(["102", "Physics", 85])
        writer.writerow(["103", "Math", 60])
        writer.writerow(["103", "Physics", 50])

    report = build_gradebook()
    for row in report:
        print(row)

{'student_id': '101', 'name': 'Haider', 'average': 85.0, 'grade': 'A', 'rank': 1}
{'student_id': '102', 'name': 'Ayesha', 'average': 85.0, 'grade': 'A', 'rank': 2}
{'student_id': '103', 'name': 'Bilal', 'average': 55.0, 'grade': 'C', 'rank': 3}
